# CCTV Safety — Reproducible YOLOv8 Baseline

This Colab notebook validates the prepared 7-class dataset, then trains YOLOv8n and YOLOv8s with the same configuration. Dataset acquisition and exhaustive annotation must be completed before running it.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import subprocess, sys, torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Dataset gate
Training is blocked when labels, metadata, group isolation, or exact-duplicate checks fail.

In [ ]:
subprocess.run([sys.executable, 'scripts/validate_dataset.py', 'dataset', '--near-duplicates'], check=True)

## Train and compare
Both model sizes use `configs/training.yaml`, the same split, augmentations, and seed. Results are written under `reports/model_comparison`.

In [ ]:
subprocess.run([sys.executable, 'scripts/train_compare.py', '--config', 'configs/training.yaml'], check=True)

In [ ]:
import json, pandas as pd
display(pd.read_csv('reports/model_comparison/summary.csv'))
for path in Path('reports/model_comparison').glob('yolov8*.json'):
    print(path.name)
    display(pd.DataFrame(json.loads(path.read_text())['per_class']).T)

## Required interpretation
Select class thresholds on validation data with Recall as the primary safety objective. Do not claim production readiness: this baseline has not been evaluated on target CCTV footage. Review false positives/negatives using `docs/evaluation.md`.